# Movement取得

通常は「設定」セルだけ編集し、上から順に実行します。\n
実行中はブラウザと進捗バーを確認できます。

In [ ]:
from pathlib import Path
import importlib
import utils_scraping_seasearcher as sss

sss = importlib.reload(sss)

In [ ]:
# ===== 設定：通常はここだけ編集 =====
# movementを取得する船舶CSV。リストの上から順に処理します。
SOURCE_FILES = [
    str(Path("vessel") / "vessels_20260817_oiltanker.csv"),
    #str(Path("vessel") / "vessels_20260817_chemicaltanker.csv"),
    #str(Path("vessel") / "vessels_20260817_liquefiedgastanker.csv"),
    #str(Path("vessel") / "vessels_20260817_otherliquidstankers.csv"),
    #str(Path("vessel") / "vessels_20260817_container.csv"),
    #str(Path("vessel") / "vessels_20260817_roro.csv"),
    #str(Path("vessel") / "vessels_20260817_bulk.csv"),
]

# 取得条件と実行方法
# SeaSearcher login account. Set both values, or leave both as None to use environment variables.
LOGIN_CONFIG = {
    "login_user": None,  # Login user
    "login_password": None,  # Login password
}

SCRAPING_CONFIG = {
    "status_list": "all",  # 選択肢: "all" / "Calls" / "Passings" / "Sightings" / ["Calls", "Sightings"]
    "period": {"from": "2025-01-01", "to": "2026-07-31"},  # from/to。日付形式: YYYY-MM-DD / YYYYMMDD / YYYY/MM/DD / DD/MM/YYYY / DD-MM-YYYY
    "local_time": False,  # 選択肢: False = GMT、True = 船舶画面のLocal Time
    "out_dir": r"movement/20250101_20260731_all_vessels",  # CSV出力先
    "login_user": LOGIN_CONFIG["login_user"],  # SeaSearcherのログインユーザー
    "login_password": LOGIN_CONFIG["login_password"],  # SeaSearcherのログインパスワード
    "headless": False,  # 選択肢: False = ブラウザを表示、True = 非表示
    "show_progress": True,  # 選択肢: True = 船舶ごとの進捗バーを表示、False = 非表示
    "log_level": "DONE",  # 選択肢: DONE / COMPLETE / INFO / WARNING / ERROR / DEBUG
    "check_status": False,  # 選択肢: False = ステータス確認なし、True = 各船のステータス確認を追加
    "skip_if_exists": True,  # 選択肢: True = 既存CSVをスキップ、False = 既存CSVも再取得
    "skip_if_known_no_data": True,  # 選択肢: True = no-data記録をスキップ、False = 再確認
    "periodic_rest_enabled": True,  # 選択肢: True = 休止時間を入れる、False = 入れない
    "work_session_hours": 6.0,  # 休止までの最大稼働時間
    "work_session_random_minutes": 20.0,  # 稼働時間に加えるランダム幅
    "rest_session_hours": 1.5,  # 休止時間
    "rest_session_random_minutes": 15.0,  # 休止時間に加えるランダム幅
    "relogin_after_timeout_streak": True,  # 5隻連続タイムアウト時だけブラウザを作り直して再ログイン
    "timeout_streak_relogin_threshold": 5,  # 再ログインを行う連続タイムアウト数
    "retry_timeout_immediately": False,  # 選択肢: False = 即時再試行なし、True = タイムアウト直後に再試行
    "retry_timeout_once_after_relogin": True,  # 再ログイン後、直前のタイムアウト船を1回だけ再試行
    "retry_recoverable_errors": False,  # 選択肢: False = 再試行なし、True = 回復可能エラー時に再ログインして再試行
}

SOURCE_FILES

In [ ]:
SOURCE_CONTEXT = sss.load_live_llinos_from_vessel_files(
    SOURCE_FILES,
    status_values=["Live"],  # 選択肢: [] = Live / Deadなど全ステータス、["Live"]や["Dead"] = 指定ステータスのみ
    unique=True,  # 選択肢: True = LLI重複を除去、False = 重複を残す
    sort=False,  # 選択肢: False = 入力順を維持、True = LLI番号順。
)

targets_to_run = SOURCE_CONTEXT["targets"]
existing_preview = sss.preview_existing_movement_outputs(
    targets_to_run,
    SCRAPING_CONFIG,
)

print(f"入力CSV: {len(SOURCE_FILES)}件")
print(f"対象LLI: {len(targets_to_run)}件")
print(f"既存またはno-data記録あり: {existing_preview['existing_count']}件")
print(f"今回処理予定: {existing_preview['pending_count']}件")
print(f"出力先: {SCRAPING_CONFIG['out_dir']}")

SOURCE_CONTEXT["source_summary"]

In [ ]:
# 実行：このセルでブラウザが開き、進捗が表示されます
results = sss.parallel_scraping(
    targets_to_run,
    max_workers=1,  # 1 = ブラウザ1つで順番に処理。サイト負荷を抑えるため1固定
    config=SCRAPING_CONFIG,
)

ok_count = sum(bool(result.get("ok")) for result in results)
failed_count = len(results) - ok_count
{"total": len(results), "ok": ok_count, "failed": failed_count}